In [ ]:
from pathlib import Path
import polars as pl
projectPath = Path.cwd().resolve()
while projectPath.parent != projectPath and not (projectPath / 'Data').exists():
    projectPath = projectPath.parent
dataPrepPath = projectPath / 'pipelines' / 'dataprep' / 'output'
primaryPath = projectPath / 'pipelines' / 'primary' / 'output'
outputPath = projectPath / 'pipelines' / 'robustness' / 'output'
outputPath.mkdir(parents=True, exist_ok=True)
evaluationRows = pl.read_parquet(dataPrepPath / 'evaluationRows.parquet')
predictions = pl.read_parquet(primaryPath / 'modelPredictions.parquet').filter(pl.col('sourceTable') == 'mlTable')
scoredRows = predictions.join(evaluationRows.select('rowId', 'ndvi', 'offsetDays', 'absOffsetDays', 'absOffsetRange', 'offsetDirection'), on='rowId', how='left').with_columns((pl.col('prediction') - pl.col('ndvi')).alias('error'))
errorBySensorOffset = scoredRows.group_by('absOffsetRange', 'offsetDirection', 'model', 'featureSet', 'seed').agg(pl.col('absOffsetDays').min().alias('minimumAbsOffsetDays'), pl.col('absOffsetDays').max().alias('maximumAbsOffsetDays'), pl.col('offsetDays').mean().alias('meanSignedOffsetDays'), pl.col('error').abs().mean().alias('mae'), pl.col('error').pow(2).mean().sqrt().alias('rmse'), pl.col('error').mean().alias('bias'), pl.len().alias('rowCount')).sort('minimumAbsOffsetDays', 'offsetDirection', 'model', 'featureSet', 'seed')
errorBySensorOffset.write_parquet(outputPath / 'errorBySensorOffset.parquet')
assert errorBySensorOffset.select(pl.col('absOffsetRange').n_unique()).item() == 3
assert errorBySensorOffset.select(pl.col('rowCount').sum()).item() == predictions.height
errorBySensorOffset
